In [0]:
from olist_lakehouse.io import read_managed_table, write_managed_table
from olist_lakehouse.transformations import parse_timestamp
from pyspark.sql.functions import (
    col,
    collect_set,
    datediff,
    when,
)
from pyspark.sql.functions import (
    max as _max,
)
from pyspark.sql.functions import (
    sum as _sum,
)

In [0]:
dbutils.widgets.text("catalog", "")

In [0]:
catalog = dbutils.widgets.get("catalog")

## Orders table
Contains a complete order information view bringing together orders, customers and payments.

**Table requirements:**

- Fields must be correctly typed with `TIMESTAMP`
- Aggregate payments by order
  - `total_payment_value` - sum of paid value
  - `max_installments` - max numbers of installments
  - `payments_types_used` - a list of distinct payment types used
- Field `delivery_delay_days`
  - difference of days between `order_delivered_customer_date` and `order_estimated_delivery_date`
  - orders not delivered must return `NULL`
- Field `is_late`
  - `TRUE` when `delivery_delay_days` > 0 

**Expected schema:**

| Field                    | Type      | Origin (bronze)                    |
| -------------------------| ----------| -----------------------------------|
| order_id                 | STRING    | olist_orders_dataset               |
| customer_unique_id       | STRING    | olist_customers_dataset            |
| customer_state           | STRING    | olist_customers_dataset            |
| order_status             | STRING    | olist_orders_dataset               |
| order_purchase_timestamp | TIMESTAMP | olist_orders_dataset               |
| total_payment_value      | DOUBLE    | olist_order_payments_dataset (agg) |
| max_installments         | INT       | olist_order_payments_dataset (agg) |
| payment_types_used       | STRING    | olist_order_payments_dataset (agg) |
| delivery_delay_days      | INT       | logic                              |
| is_late                  | BOOLEAN   | logic                              |

In [0]:
orders = read_managed_table(
    catalog=catalog,
    schema="bronze",
    table="olist_orders_dataset",
)
customers = read_managed_table(
    catalog=catalog,
    schema="bronze",
    table="olist_customers_dataset",
)
payments = read_managed_table(
    catalog=catalog,
    schema="bronze",
    table="olist_order_payments_dataset",
)

In [0]:
payments = (
    payments
    .withColumn("payment_value", col("payment_value").cast("double"))
    .withColumn("payment_installments", col("payment_installments").cast("int"))
)

payments_agg = (
    payments
    .groupBy("order_id")
    .agg(
        _sum(col("payment_value")).alias("total_payment_value"),
        _max(col("payment_installments")).alias("max_installments"),
        collect_set(col("payment_type")).alias("payment_types_used"),
    )
    .withColumn("payment_types_used", col("payment_types_used").cast("string"))
)

In [0]:
orders_join = orders.join(customers, on="customer_id", how="left").join(payments_agg, on="order_id", how="left")
orders_transformed = orders_join.select(
    "order_id",
    "customer_unique_id",
    "customer_state",
    "order_status",
    parse_timestamp("order_purchase_timestamp").alias("order_purchase_timestamp"),
    "total_payment_value",
    "max_installments",
    "payment_types_used",
    when(col("order_delivered_customer_date").isNull(), None)
    .otherwise(
        datediff(
            parse_timestamp("order_delivered_customer_date"),
            parse_timestamp("order_estimated_delivery_date"),
        )
    )
    .alias("delivery_delay_days"),
).withColumn("is_late", when(col("delivery_delay_days") > 0, True).otherwise(False))

In [0]:
write_managed_table(
    orders_transformed,
    mode="overwrite",
    table_full_path=f"{catalog}.silver.orders",
    options={"overwriteSchema": "true"}
)